In [51]:
import requests
import folium
from folium.plugins import MarkerCluster

# Get earthquake data
url = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson"
response = requests.get(url)
data = response.json()

# Create map centered on Asia
m = folium.Map(location=[25,100], zoom_start=4)

# Marker clustering
cluster = MarkerCluster().add_to(m)

# Asia boundary filter
asia_bounds = {
    "lat_min": -10,
    "lat_max": 60,
    "lon_min": 60,
    "lon_max": 150
}

for eq in data["features"]:

    coords = eq["geometry"]["coordinates"]
    lon = coords[0]
    lat = coords[1]

    # Filter only Asia
    if not (asia_bounds["lat_min"] <= lat <= asia_bounds["lat_max"] and
            asia_bounds["lon_min"] <= lon <= asia_bounds["lon_max"]):
        continue

    mag = eq["properties"]["mag"]
    place = eq["properties"]["place"]

    if mag is None:
        continue

    # Color by magnitude
    if mag < 3:
        color = "green"
    elif mag < 5:
        color = "orange"
    else:
        color = "red"

    folium.CircleMarker(
        location=[lat, lon],
        radius=mag * 2,
        popup=f"<b>{place}</b><br>Magnitude: {mag}",
        color=color,
        fill=True
    ).add_to(cluster)

# Dashboard title
title_html = '''
<div style="
position: fixed;
top: 10px;
left: 50%;
transform: translateX(-50%);
z-index: 9999;
background-color: white;
padding: 12px 20px;
border-radius: 10px;
box-shadow: 0 2px 8px rgba(0,0,0,0.2);
font-family: Arial;
">
<h3 style="margin:0;">🌍 Asia Earthquake Intelligence Dashboard</h3>
<p style="margin:0;font-size:13px;">
Author: Abdul Subhan | Data Source: USGS
</p>
</div>
'''

m.get_root().html.add_child(folium.Element(title_html))

# Legend
legend = '''
<div style="
position: fixed;
bottom: 50px;
left: 50px;
z-index:9999;
background-color:white;
padding:10px;
border-radius:8px;
font-size:14px;
">
<b>Magnitude Legend</b><br>
🟢 Green : <3<br>
🟠 Orange : 3–5<br>
🔴 Red : >5
</div>
'''

m.get_root().html.add_child(folium.Element(legend))

# Save dashboard
m.save("earthquake_dashboard.html")

m